<a href="https://colab.research.google.com/github/Sujitha519/AgriMatch_AI/blob/main/AgriMatch_AI_Production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# STEP 1: INSTALL & IMPORT DEPENDENCIES
# ==============================================================================
!pip install -q scikit-learn pandas numpy gradio

import datetime
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import gradio as gr

print("[✓] Dependencies successfully loaded.")

# ==============================================================================
# STEP 2: DATASET GENERATION & MODEL TRAINING
# ==============================================================================
np.random.seed(42)
dates = pd.date_range(start="2023-01-01", end="2026-08-01", freq="D")
crops = ['Tomato', 'Potato', 'Onion', 'Rice', 'Wheat']
regions = ['North Mandi', 'South Market', 'Central Hub', 'Eastern Port']

data = []
for date in dates:
    for crop in crops:
        for region in regions:
            month = date.month
            seasonality = np.sin(2 * np.pi * month / 12) * 150
            base_price = {'Tomato': 1800, 'Potato': 1200, 'Onion': 2200, 'Rice': 3500, 'Wheat': 2400}[crop]

            rainfall_mm = np.random.uniform(0, 50) + (100 if month in [6,7,8] else 10)
            demand_index = np.random.uniform(0.7, 1.5)

            modal_price = max(500, base_price + seasonality + (demand_index * 300) - (rainfall_mm * 2) + np.random.normal(0, 80))

            data.append({
                'Date': date,
                'Crop': crop,
                'Region': region,
                'Month': month,
                'DayOfWeek': date.dayofweek,
                'Rainfall_mm': rainfall_mm,
                'Demand_Index': demand_index,
                'Price_Per_Quintal': round(modal_price, 2)
            })

df = pd.DataFrame(data)
df_encoded = pd.get_dummies(df, columns=['Crop', 'Region'], drop_first=False)

X = df_encoded.drop(columns=['Date', 'Price_Per_Quintal'])
y = df_encoded['Price_Per_Quintal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("[✓] Model Training Complete!")

# ==============================================================================
# STEP 3: DECISION ENGINE LOGIC
# ==============================================================================
BUYERS_DB = [
    {"Name": "AgriCorp Processing Ltd", "Region": "North Mandi", "Distance_km": 25, "Type": "Food Processor", "Price_Premium": 1.08},
    {"Name": "FreshVeg Supermarket Chain", "Region": "South Market", "Distance_km": 60, "Type": "Retailer", "Price_Premium": 1.15},
    {"Name": "Central Wholesale Terminal", "Region": "Central Hub", "Distance_km": 15, "Type": "Wholesaler", "Price_Premium": 1.02},
    {"Name": "Global Export Logistics", "Region": "Eastern Port", "Distance_km": 120, "Type": "Exporter", "Price_Premium": 1.25},
]

def predict_price_and_match_buyers(crop, region, quantity_quintals, forecast_days, transport_cost_per_km):
    try:
        # Safe Type Conversions
        qty = float(quantity_quintals) if quantity_quintals else 20.0
        days = int(forecast_days) if forecast_days else 7
        t_cost = float(transport_cost_per_km) if transport_cost_per_km else 30.0
        selected_crop = str(crop) if crop else 'Tomato'
        selected_region = str(region) if region else 'North Mandi'

        future_date = datetime.date.today() + datetime.timedelta(days=days)

        # Build Input Vector
        input_data = {
            'Month': future_date.month,
            'DayOfWeek': future_date.weekday(),
            'Rainfall_mm': 15.0,
            'Demand_Index': 1.1
        }

        for col in X.columns:
            if col.startswith('Crop_'):
                input_data[col] = 1.0 if col == f"Crop_{selected_crop}" else 0.0
            elif col.startswith('Region_'):
                input_data[col] = 1.0 if col == f"Region_{selected_region}" else 0.0

        input_df = pd.DataFrame([input_data])[X.columns]
        predicted_market_price = float(model.predict(input_df)[0])

        matches = []
        for b in BUYERS_DB:
            gross_rate = predicted_market_price * b['Price_Premium']
            gross_revenue = gross_rate * qty
            transport_cost = b['Distance_km'] * t_cost
            net_profit = gross_revenue - transport_cost
            net_rate_per_quintal = net_profit / qty if qty > 0 else 0

            matches.append({
                "Buyer Name": b['Name'],
                "Buyer Type": b['Type'],
                "Distance (km)": b['Distance_km'],
                "Offered Rate (₹/Qtl)": round(gross_rate, 2),
                "Estimated Transport (₹)": round(transport_cost, 2),
                "Net Profit (₹)": round(net_profit, 2),
                "Effective Net Rate (₹/Qtl)": round(net_rate_per_quintal, 2)
            })

        match_df = pd.DataFrame(matches).sort_values(by="Net Profit (₹)", ascending=False)
        best_buyer = match_df.iloc[0]

        summary = f"""
### 📈 **AI Market Forecast Analysis**
* **Target Commodity:** `{selected_crop}` | **Base District:** `{selected_region}`
* **Forecast Horizon:** `{future_date.strftime('%d %B %Y')}` ({days} Days Ahead)
* **Predicted Mandi Base Rate:** **₹{predicted_market_price:.2f} / Quintal**

---
### 🏆 **Optimal Direct Buyer Recommendation**
* **Recommended Channel:** **{best_buyer['Buyer Name']}** ({best_buyer['Buyer Type']})
* **Total Net Profit:** **₹{best_buyer['Net Profit (₹)']:,.2f}** (after deducting ₹{best_buyer['Estimated Transport (₹)']} logistics costs)
* **Effective Payout Rate:** **₹{best_buyer['Effective Net Rate (₹/Qtl)']:.2f} / Quintal**
"""
        return summary, match_df

    except Exception as e:
        return f"Error executing model calculation: {str(e)}", pd.DataFrame()

# ==============================================================================
# STEP 4: GRADIO WEB LAUNCHER
# ==============================================================================
demo = gr.Interface(
    fn=predict_price_and_match_buyers,
    inputs=[
        gr.Dropdown(choices=['Tomato', 'Potato', 'Onion', 'Rice', 'Wheat'], value='Tomato', label="Select Commodity / Crop"),
        gr.Dropdown(choices=['North Mandi', 'South Market', 'Central Hub', 'Eastern Port'], value='North Mandi', label="District/Region"),
        gr.Number(value=20, label="Quantity (Quintals)"),
        gr.Slider(minimum=1, maximum=30, value=7, step=1, label="Forecast Horizon (Days)"),
        gr.Number(value=30, label="Transport Cost (₹/km)")
    ],
    outputs=[
        gr.Markdown(label="AI Strategy Summary"),
        gr.Dataframe(label="Optimal Buyer Matching Matrix")
    ],
    title="🌾 AgriMatch AI: Market Intelligence & Buyer Matching",
    description="Select your crop parameters to forecast prices and find the highest net-profit direct buyers.",
    theme="emerald"
)

demo.launch(share=True, debug=True)

[✓] Dependencies successfully loaded.
[✓] Model Training Complete!


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:583: UserWarning: Cannot load emerald. Caught Exception: Client error '404 Not Found' for url 'https://huggingface.co/api/spaces/emerald' (Request ID: Root=1-6a78ba93-3db8dbad537d9c5c034e5cf8;a151f92e-71fd-4ebe-8545-bf8379c1f8e7)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://98060e54980bb35970.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
